# 02 — Feature Engineering
Analyse des features disponibles et sélection des 30 plus discriminantes pour détecter les bots.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_parquet('../data/twitter_clean.parquet')
print(df.shape)
df.head()

## Catégorie 1 — Activité du compte
`statuses_count`, `favourites_count`, `listed_count`, `account_age`

In [ ]:
print('Tweets publiés — médiane humains :', df[df['class_bot']==0]['statuses_count'].median())
print('Tweets publiés — médiane bots    :', df[df['class_bot']==1]['statuses_count'].median())

In [ ]:
print('Âge du compte — médiane humains :', df[df['class_bot']==0]['account_age'].median(), 'jours')
print('Âge du compte — médiane bots    :', df[df['class_bot']==1]['account_age'].median(), 'jours')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['statuses_count', 'account_age']):
    df.groupby('class_bot')[col].median().plot.bar(ax=ax, color=['steelblue', 'crimson'])
    ax.set_xticklabels(['Humain', 'Bot'], rotation=0)
    ax.set_title(f'Médiane — {col}')
plt.tight_layout()
plt.show()

## Catégorie 2 — Réseau social
`followers_count`, `friends_count`, `friends_followers_ratio`

Le ratio abonnements/followers est le signal le plus discriminant : un bot suit massivement des comptes sans en attirer.

In [ ]:
print('Ratio following/followers — médiane humains :', round(df[df['class_bot']==0]['friends_followers_ratio'].median(), 2))
print('Ratio following/followers — médiane bots    :', round(df[df['class_bot']==1]['friends_followers_ratio'].median(), 2))

In [ ]:
df.groupby('class_bot')['friends_followers_ratio'].median().plot.bar(color=['steelblue', 'crimson'])
plt.xticks([0, 1], ['Humain', 'Bot'], rotation=0)
plt.title('Médiane du ratio abonnements/followers')
plt.tight_layout()
plt.show()

## Catégorie 3 — Identité du profil
`default_profile`, `default_profile_image`, `description_length`, `name_entropy`

Les bots ont souvent une photo par défaut et aucune biographie.

In [ ]:
n_bots = df[df['class_bot']==1]
print(f'Bots avec photo par défaut : {int(n_bots["default_profile_image"].sum())} / {len(n_bots)}')
print(f'Bots avec profil par défaut : {int(n_bots["default_profile"].sum())} / {len(n_bots)}')

In [ ]:
identity_cols = ['default_profile', 'default_profile_image', 'name_contains_bot', 'screen_name_contains_bot']
df.groupby('class_bot')[identity_cols].mean().T.plot.bar(figsize=(10, 4), color=['steelblue', 'crimson'])
plt.title('Taux moyen — features identité')
plt.xticks(rotation=30)
plt.legend(['Humain', 'Bot'])
plt.tight_layout()
plt.show()

## Catégorie 4 — Ratios temporels
`statuses_account_age_ratio`, `followers_account_age_ratio`, `friends_account_age_ratio`

Normaliser par l'âge du compte permet de détecter les bots récents très actifs.

In [ ]:
print('Tweets/jour — médiane humains :', round(df[df['class_bot']==0]['statuses_account_age_ratio'].median(), 5))
print('Tweets/jour — médiane bots    :', round(df[df['class_bot']==1]['statuses_account_age_ratio'].median(), 5))

In [ ]:
ratio_cols = ['statuses_account_age_ratio', 'followers_account_age_ratio', 'friends_account_age_ratio']
df.groupby('class_bot')[ratio_cols].median().T.plot.bar(figsize=(10, 4), color=['steelblue', 'crimson'])
plt.title('Médiane des ratios temporels')
plt.xticks(rotation=30)
plt.legend(['Humain', 'Bot'])
plt.tight_layout()
plt.show()

## Corrélation des features avec le label

In [ ]:
FEATURES = [
    'statuses_count', 'followers_count', 'friends_count', 'favourites_count',
    'listed_count', 'default_profile', 'default_profile_image', 'geo_enabled',
    'profile_use_background_image', 'verified',
    'name_length', 'screen_name_length', 'description_length',
    'name_contains_bot', 'screen_name_contains_bot',
    'name_entropy', 'screen_name_entropy',
    'friends_followers_ratio', 'lists_followers_ratio',
    'retweet_followers_ratio', 'favorites_followers_ratio',
    'retweet_status_ratio', 'favorites_status_ratio', 'reply_status_ratio',
    'account_age',
    'followers_account_age_ratio', 'friends_account_age_ratio',
    'statuses_account_age_ratio', 'favourites_account_age_ratio',
    'lists_account_age_ratio',
]
print(f'{len(FEATURES)} features sélectionnées')

In [ ]:
corr = df[FEATURES + ['class_bot']].corr()['class_bot'].drop('class_bot').sort_values()
plt.figure(figsize=(8, 10))
corr.plot.barh(color=['crimson' if v > 0 else 'steelblue' for v in corr])
plt.title('Corrélation avec le label (bot = 1)')
plt.axvline(0, color='black', lw=0.8)
plt.tight_layout()
plt.savefig('../plots/feature_correlation.png', dpi=120)
plt.show()

In [ ]:
top3 = corr.abs().sort_values(ascending=False).head(3)
print('Top 3 features les plus corrélées avec le label :')
print(top3)